In [ ]:
!pip install tensorflow
!pip install datasets

In [ ]:
import numpy as np
from datasets import load_dataset
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Bidirectional
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
text = " ".join(ds['train']['text'])
text = text.replace('\n', ' ').strip()
print(f"Text length: {len(text)} characters")

Text length: 10929702 characters


In [ ]:
chars = sorted(list(set(text)))
mapping = {char: idx for idx, char in enumerate(chars)}
vocab_size = len(mapping)
print(f"Vocabulary size (unique characters): {vocab_size}")

Vocabulary size (unique characters): 1012


In [ ]:
seq_length = 100
sequences = []
for i in range(seq_length, len(text)):
    seq = text[i-seq_length:i+1]
    encoded = [mapping[char] for char in seq]
    sequences.append(encoded)

print(f"Total sequences: {len(sequences)}")

Total sequences: 10929602


In [ ]:
sequences = np.array(sequences)
X, y = sequences[:, :-1], sequences[:, -1]

In [ ]:
X = X[:50000]
y = y[:50000]

In [ ]:
def create_model(vocab_size, seq_length):
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=128, input_length=seq_length))
    model.add(Bidirectional(LSTM(256, return_sequences=True)))
    model.add(Bidirectional(LSTM(256)))
    model.add(Dense(vocab_size, activation='softmax'))
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.summary()
    return model

model = create_model(vocab_size, X.shape[1])
model.fit(X, y, epochs=30, batch_size=128)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
 39/391 ━━━━━━━━━━━━━━━━━━━━ 59:03 10s/step - accuracy: 0.1638 - loss: 4.7497

KeyboardInterrupt: 

In [ ]:
def generate_sequence(model, mapping, max_length, seed_text, n_chars):
    inverse_mapping = {v: k for k, v in mapping.items()}
    text = seed_text
    for _ in range(n_chars):
        encoded = [mapping[char] for char in text[-max_length:]]
        encoded = pad_sequences([encoded], maxlen=max_length, truncating='pre')
        # Remove the to_categorical conversion
        # encoded = to_categorical(encoded, num_classes=vocab_size)
        yhat = model.predict(encoded, verbose=0)[0]
        yhat = sample(yhat, temperature=0.7)  # try values like 0.7, 1.0, 1.2

        def sample(preds, temperature=1.0):
          preds = np.asarray(preds).astype("float64")
          preds = np.log(preds + 1e-8) / temperature
          exp_preds = np.exp(preds)
          preds = exp_preds / np.sum(exp_preds)
          probas = np.random.multinomial(1, preds, 1)
          return np.argmax(probas)

        text += inverse_mapping[yhat]
    return text

print(generate_sequence(model, mapping, seq_length, "the scientific method involves", 300))

The scientific method is the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the and the
